# RI-JK Skeleton 二阶导数优化

In [2]:
from pyscf import gto, scf, lib
import numpy as np
import scipy
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [3]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [4]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [5]:
de_J20 = np.load("nh3_r_hf_decomp.npz")["de_J20"]
de_J11 = np.load("nh3_r_hf_decomp.npz")["de_J11"]
de_J02 = np.load("nh3_r_hf_decomp.npz")["de_J02"]
de_K20 = np.load("nh3_r_hf_decomp.npz")["de_K20"]
de_K11 = np.load("nh3_r_hf_decomp.npz")["de_K11"]
de_K02 = np.load("nh3_r_hf_decomp.npz")["de_K02"]

In [6]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)

# 原理性分解

下面的分解与 02-*.ipynb 的一些单元非常接近。我们将用这些单元作为验证的参考。

In [7]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])
int3c2e_ipip1 = _int3c_wrapper(mol, aux, "int3c2e_ipip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipvip1 = _int3c_wrapper(mol, aux, "int3c2e_ipvip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ip1ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip1ip2", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipip2 = _int3c_wrapper(mol, aux, "int3c2e_ipip2", "s1")().reshape([3, 3, nao, nao, naux])
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int2c2e_ipip1 = aux.intor("int2c2e_ipip1").reshape([3, 3, naux, naux])
int2c2e_ip1ip2 = aux.intor("int2c2e_ip1ip2").reshape([3, 3, naux, naux])


### J (basis_2nd)

In [8]:
# (10|0)(0|10)
dbas_J20_1 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, dm0, dm0)
de_J20_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_J20_1[A, B] += 4 * np.einsum("tsuv -> ts", dbas_J20_1[:, :, p0A:p1A, p0B:p1B])

In [9]:
# (11|0)(0|00)
dbas_J20_2 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsuv", int3c2e_ipvip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J20_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_J20_2[A, B] += 2 * np.einsum("tsuv -> ts", dbas_J20_2[:, :, p0A:p1A, p0B:p1B])

In [10]:
# (20|0)(0|00)
dbas_J20_3 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsuv", int3c2e_ipip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J20_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    de_J20_3[A, A] += 2 * np.einsum("tsuv -> ts", dbas_J20_3[:, :, p0A:p1A])

In [11]:
de_J20_recap = de_J20_1 + de_J20_2 + de_J20_3
assert np.allclose(de_J20_recap, de_J20)

### J (basis_1st ux_1st)

In [12]:
# (10|1)(0|0)(0|00)
dbas_J11_1 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsuP", int3c2e_ip1ip2, int2c2e_inv, int3c2e, dm0, dm0)
de_J11_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_1[A, B] += 2 * np.einsum("tsuP -> ts", dbas_J11_1[:, :, p0A:p1A, p0B:p1B])
de_J11_1 += de_J11_1.transpose(1, 0, 3, 2)

In [13]:
# (10|0)(0|1)(0|00)
dbas_J11_2 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsuR", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J11_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_2[A, B] += 2 * np.einsum("tsuR -> ts", dbas_J11_2[:, :, p0A:p1A, p0B:p1B])
de_J11_2 += de_J11_2.transpose(1, 0, 3, 2)

In [14]:
# (10|0)(1|0)(0|00)
dbas_J11_3 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsuQ", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J11_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_3[A, B] += -2 * np.einsum("tsuQ -> ts", dbas_J11_3[:, :, p0A:p1A, p0B:p1B])
de_J11_3 += de_J11_3.transpose(1, 0, 3, 2)

In [ ]:
# (10|0)(0|0)(1|00)
dbas_J11_4 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsuQ", int3c2e_ip1, int2c2e_inv, int3c2e_ip2, dm0, dm0)
de_J11_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J11_4[A, B] += 2 * np.einsum("tsuQ -> ts", dbas_J11_4[:, :, p0A:p1A, p0B:p1B])
de_J11_4 += de_J11_4.transpose(1, 0, 3, 2)

In [16]:
de_J11_recap = de_J11_1 + de_J11_2 + de_J11_3 + de_J11_4
assert np.allclose(de_J11_recap, de_J11)

### J (aux_2nd)

In [17]:
# (00|2)(0|00)
dbas_J02_1 = np.einsum("tsuvP, PQ, klQ, uv, kl -> tsP", int3c2e_ipip2, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_J02_1[A, A] += np.einsum("tsP -> ts", dbas_J02_1[:, :, p0A:p1A])

In [18]:
# (00|0)(2|0)(0|00)
dbas_J02_2 = np.einsum("uvP, PQ, tsQR, RS, klS, uv, kl -> tsQ", int3c2e, int2c2e_inv, int2c2e_ipip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_J02_2[A, A] += -1 * np.einsum("tsQ -> ts", dbas_J02_2[:, :, p0A:p1A])
de_J02_2 = de_J02_2

In [19]:
# (00|0)(1|1)(0|00)
dbas_J02_3a = np.einsum("uvP, PQ, tsQR, RS, klS, uv, kl -> tsQR", int3c2e, int2c2e_inv, int2c2e_ip1ip2, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_3a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_3a[A, B] += -0.5 * np.einsum("tsQR -> ts", dbas_J02_3a[:, :, p0A:p1A, p0B:p1B])
de_J02_3a += de_J02_3a.transpose(1, 0, 3, 2)

In [20]:
# (00|0)(1|0)(0|1)(0|00)
dbas_J02_3b = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, uv, kl -> tsQT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_3b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_3b[A, B] += -0.5 * np.einsum("tsQT -> ts", dbas_J02_3b[:, :, p0A:p1A, p0B:p1B])
de_J02_3b += de_J02_3b.transpose(1, 0, 3, 2)

In [21]:
# (00|1)(1|0)(0|00)
dbas_J02_4 = np.einsum("tuvP, PQ, sQR, RS, klS, uv, kl -> tsPQ", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_4[A, B] += -1 * np.einsum("tsPQ -> ts", dbas_J02_4[:, :, p0A:p1A, p0B:p1B])
de_J02_4 += de_J02_4.transpose(1, 0, 3, 2)

In [22]:
# (00|1)(1|00)
dbas_J02_5 = np.einsum("tuvP, PQ, sklQ, uv, kl -> tsPQ", int3c2e_ip2, int2c2e_inv, int3c2e_ip2, dm0, dm0)
de_J02_5 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_5[A, B] += 0.5 * np.einsum("tsPQ -> ts", dbas_J02_5[:, :, p0A:p1A, p0B:p1B])
de_J02_5 += de_J02_5.transpose(1, 0, 3, 2)

In [23]:
# (00|0)(0|1)(1|0)(0|00)
dbas_J02_6 = np.einsum("uvP, PQ, tRQ, RS, sST, TU, klU, uv, kl -> tsRS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_6 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_6[A, B] += 0.5 * np.einsum("tsRS -> ts", dbas_J02_6[:, :, p0A:p1A, p0B:p1B])
de_J02_6 += de_J02_6.transpose(1, 0, 3, 2)

In [24]:
# (00|1)(0|1)(0|00)
dbas_J02_7 = np.einsum("tuvP, PQ, sRQ, RS, klS, uv, kl -> tsPR", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_7 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_7[A, B] += -1 * np.einsum("tsPR -> ts", dbas_J02_7[:, :, p0A:p1A, p0B:p1B])
de_J02_7 += de_J02_7.transpose(1, 0, 3, 2)

In [25]:
# (00|0)(1|0)(1|0)(0|00)
dbas_J02_8 = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, uv, kl -> tsRT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, dm0, dm0)
de_J02_8 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_J02_8[A, B] += 1 * np.einsum("tsRT -> ts", dbas_J02_8[:, :, p0A:p1A, p0B:p1B])
de_J02_8 += de_J02_8.transpose(1, 0, 3, 2)

In [26]:
de_J02_recap = de_J02_1 + de_J02_2 + de_J02_3a + de_J02_3b + de_J02_4 + de_J02_5 + de_J02_6 + de_J02_7 + de_J02_8
assert np.allclose(de_J02_recap, de_J02, atol=1e-5, rtol=1e-4)

### K (basis_2nd)

In [27]:
# (10|0)(0|10), part a
dbas_K20_1a = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_1a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_1a[A, B] += 2 * np.einsum("tsuk -> ts", dbas_K20_1a[:, :, p0A:p1A, p0B:p1B])

In [28]:
# (10|0)(0|10), part b
dbas_K20_1b = np.einsum("tuvP, PQ, sklQ, ui, vj, kj, li -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_1b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_1b[A, B] += 2 * np.einsum("tsuk -> ts", dbas_K20_1b[:, :, p0A:p1A, p0B:p1B])

In [29]:
# (11|0)(0|00)
dbas_K20_2 = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsuv", int3c2e_ipvip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_2[A, B] += 2 * np.einsum("tsuv -> ts", dbas_K20_2[:, :, p0A:p1A, p0B:p1B])

In [30]:
# (20|0)(0|00)
dbas_K20_3 = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsuv", int3c2e_ipip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    de_K20_3[A, A] += 2 * np.einsum("tsuv -> ts", dbas_K20_3[:, :, p0A:p1A])

In [31]:
de_K20_recap = de_K20_1a + de_K20_1b + de_K20_2 + de_K20_3
assert np.allclose(de_K20_recap, de_K20)

### K (basis_1st_aux_1st)

In [32]:
# (10|1)(0|0)(0|00)
dbas_K11_1 = np.einsum("tsuvP, PQ, klQ, vi, li, uj, kj -> tsuP", int3c2e_ip1ip2, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_1[A, B] += 2 * np.einsum("tsuP -> ts", dbas_K11_1[:, :, p0A:p1A, p0B:p1B])
de_K11_1 += de_K11_1.transpose(1, 0, 3, 2)

In [33]:
# (10|0)(0|1)(0|00)
dbas_K11_2 = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsuR", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_2[A, B] += 2 * np.einsum("tsuR -> ts", dbas_K11_2[:, :, p0A:p1A, p0B:p1B])
de_K11_2 += de_K11_2.transpose(1, 0, 3, 2)

In [34]:
# (10|0)(1|0)(0|00)
dbas_K11_3 = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsuQ", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_3[A, B] += -2 * np.einsum("tsuQ -> ts", dbas_K11_3[:, :, p0A:p1A, p0B:p1B])
de_K11_3 += de_K11_3.transpose(1, 0, 3, 2)

In [35]:
# (10|0)(0|0)(1|00)
dbas_K11_4 = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsuQ", int3c2e_ip1, int2c2e_inv, int3c2e_ip2, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_4[A, B] += 2 * np.einsum("tsuQ -> ts", dbas_K11_4[:, :, p0A:p1A, p0B:p1B])
de_K11_4 += de_K11_4.transpose(1, 0, 3, 2)

In [36]:
de_K11_recap = de_K11_1 + de_K11_2 + de_K11_3 + de_K11_4
assert np.allclose(de_K11_recap, de_K11)

### K (aux_2nd)

In [37]:
# (00|2)(0|00)
dbas_K02_1 = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsP", int3c2e_ipip2, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_K02_1[A, A] += np.einsum("tsP -> ts", dbas_K02_1[:, :, p0A:p1A])

In [38]:
# (00|0)(2|0)(0|00)
dbas_K02_2 = np.einsum("uvP, PQ, tsQR, RS, klS, ui, vj, ki, lj -> tsQ", int3c2e, int2c2e_inv, int2c2e_ipip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_K02_2[A, A] += -1 * np.einsum("tsQ -> ts", dbas_K02_2[:, :, p0A:p1A])
de_K02_2 = de_K02_2

In [39]:
# (00|0)(1|1)(0|00)
dbas_K02_3a = np.einsum("uvP, PQ, tsQR, RS, klS, ui, vj, ki, lj -> tsQR", int3c2e, int2c2e_inv, int2c2e_ip1ip2, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_3a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_3a[A, B] += -0.5 * np.einsum("tsQR -> ts", dbas_K02_3a[:, :, p0A:p1A, p0B:p1B])
de_K02_3a += de_K02_3a.transpose(1, 0, 3, 2)

In [40]:
# (00|0)(1|0)(0|1)(0|00)
dbas_K02_3b = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, ui, vj, ki, lj -> tsQT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_3b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_3b[A, B] += -0.5 * np.einsum("tsQT -> ts", dbas_K02_3b[:, :, p0A:p1A, p0B:p1B])
de_K02_3b += de_K02_3b.transpose(1, 0, 3, 2)

In [41]:
# (00|1)(1|0)(0|00)
dbas_K02_4 = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsPQ", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_4[A, B] += -1 * np.einsum("tsPQ -> ts", dbas_K02_4[:, :, p0A:p1A, p0B:p1B])
de_K02_4 += de_K02_4.transpose(1, 0, 3, 2)

In [42]:
# (00|1)(1|00)
dbas_K02_5 = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsPQ", int3c2e_ip2, int2c2e_inv, int3c2e_ip2, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_5 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_5[A, B] += 0.5 * np.einsum("tsPQ -> ts", dbas_K02_5[:, :, p0A:p1A, p0B:p1B])
de_K02_5 += de_K02_5.transpose(1, 0, 3, 2)

In [43]:
# (00|0)(0|1)(1|0)(0|00)
dbas_K02_6 = np.einsum("uvP, PQ, tRQ, RS, sST, TU, klU, ui, vj, ki, lj -> tsRS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_6 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_6[A, B] += 0.5 * np.einsum("tsRS -> ts", dbas_K02_6[:, :, p0A:p1A, p0B:p1B])
de_K02_6 += de_K02_6.transpose(1, 0, 3, 2)

In [44]:
# (00|1)(0|1)(0|00)
dbas_K02_7 = np.einsum("tuvP, PQ, sRQ, RS, klS, ui, vj, ki, lj -> tsPR", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_7 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_7[A, B] += -1 * np.einsum("tsPR -> ts", dbas_K02_7[:, :, p0A:p1A, p0B:p1B])
de_K02_7 += de_K02_7.transpose(1, 0, 3, 2)

In [45]:
# (00|0)(1|0)(1|0)(0|00)
dbas_K02_8 = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, ui, vj, ki, lj -> tsQS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_8 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_8[A, B] += 1 * np.einsum("tsQS -> ts", dbas_K02_8[:, :, p0A:p1A, p0B:p1B])
de_K02_8 += de_K02_8.transpose(1, 0, 3, 2)

In [46]:
de_K02_recap = de_K02_1 + de_K02_2 + de_K02_3a + de_K02_3b + de_K02_4 + de_K02_5 + de_K02_6 + de_K02_7 + de_K02_8
assert np.allclose(de_K02_recap, de_K02, atol=1e-5, rtol=1e-4)

### J1ao

In [47]:
scr1 = np.einsum("tuvP, PQ, klQ, kl -> tuv", int3c2e_ip1, int2c2e_inv, int3c2e, dm0)

j1ao_aux0 = np.zeros([natm, 3, nao, nao])
j1ao_aux0_1 = np.zeros([natm, 3, nao, nao])
j1ao_aux0_2 = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    # (10|0)(0|00)
    j1ao_aux0_1[A, :, slcA, :] -= scr1[:, slcA, :]
    # (01|0)(0|00) (can be symmetrized)
    j1ao_aux0_1[A, :, :, slcA] -= scr1[:, slcA, :].swapaxes(-1, -2)
    # (00|0)(0|10), (00|0)(0|01)
    scr2 = np.einsum("tklP, PQ, uvQ, kl -> tuv", int3c2e_ip1[:, slcA], int2c2e_inv, int3c2e, dm0[slcA])
    j1ao_aux0_2[A] -= 2 * scr2
j1ao_aux0 = j1ao_aux0_1 + j1ao_aux0_2

In [48]:
j1ao_aux1 = np.zeros([natm, 3, nao, nao])
j1ao_aux1_1 = np.zeros([natm, 3, nao, nao])
j1ao_aux1_2 = np.zeros([natm, 3, nao, nao])
j1ao_aux1_3 = np.zeros([natm, 3, nao, nao])
j1ao_aux1_4 = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    # (00|1)(0|00)
    j1ao_aux1_1[A] -= np.einsum("tuvP, PQ, klQ, kl -> tuv", int3c2e_ip2[:, :, :, slcA], int2c2e_inv[slcA, :], int3c2e, dm0)
    # (00|0)(1|00)
    j1ao_aux1_2[A] -= np.einsum("uvP, PQ, tklQ, kl -> tuv", int3c2e, int2c2e_inv[:, slcA], int3c2e_ip2[:, :, :, slcA], dm0)
    # (00|0)(1|0)(0|00)
    j1ao_aux1_3[A] += np.einsum("uvP, PQ, tQR, RS, klS, kl -> tuv", int3c2e, int2c2e_inv[:, slcA], int2c2e_ip1[:, slcA], int2c2e_inv, int3c2e, dm0)
    # (00|0)(0|1)(0|00)
    j1ao_aux1_4[A] += np.einsum("uvP, PQ, tRQ, RS, klS, kl -> tuv", int3c2e, int2c2e_inv, int2c2e_ip1[:, slcA], int2c2e_inv[slcA, :], int3c2e, dm0)
j1ao_aux1 = j1ao_aux1_1 + j1ao_aux1_2 + j1ao_aux1_3 + j1ao_aux1_4

### K1ao

In [49]:
scr1 = np.einsum("tuvP, PQ, klQ, vi, li -> tuk", int3c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2)

k1ao_aux0 = np.zeros([natm, 3, nao, nao])
k1ao_aux0_1 = np.zeros([natm, 3, nao, nao])
k1ao_aux0_2 = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    # (10|0)(0|00)
    k1ao_aux0_1[A, :, slcA, :] -= scr1[:, slcA, :]
    # (01|0)(0|00)
    k1ao_aux0_1[A, :, :, slcA] -= scr1[:, slcA, :].swapaxes(-1, -2)
    # (00|0)(0|10), (00|0)(0|01)
    scr2 = np.einsum("tklP, PQ, uvQ, ki, ui -> tlv", int3c2e_ip1[:, slcA], int2c2e_inv, int3c2e, mocc_2[slcA], mocc_2)
    k1ao_aux0_2[A] -= scr2 + scr2.swapaxes(-1, -2)
k1ao_aux0 = k1ao_aux0_1 + k1ao_aux0_2

In [50]:
# this part of computation is not sutiable using dm-only when einsum
k1ao_aux1 = np.zeros([natm, 3, nao, nao])
k1ao_aux1_1 = np.zeros([natm, 3, nao, nao])
k1ao_aux1_2 = np.zeros([natm, 3, nao, nao])
k1ao_aux1_3 = np.zeros([natm, 3, nao, nao])
k1ao_aux1_4 = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    # (00|1)(0|00)
    k1ao_aux1_1[A] -= np.einsum("tuvP, PQ, klQ, vi, li -> tuk", int3c2e_ip2[:, :, :, slcA], int2c2e_inv[slcA, :], int3c2e, mocc_2, mocc_2)
    # (00|0)(1|00)
    k1ao_aux1_2[A] -= np.einsum("uvP, PQ, tklQ, vi, li -> tuk", int3c2e, int2c2e_inv[:, slcA], int3c2e_ip2[:, :, :, slcA], mocc_2, mocc_2)
    # (00|0)(1|0)(0|00)
    k1ao_aux1_3[A] += np.einsum("uvP, PQ, tQR, RS, klS, vi, li -> tuk", int3c2e, int2c2e_inv[:, slcA], int2c2e_ip1[:, slcA], int2c2e_inv, int3c2e, mocc_2, mocc_2)
    # (00|0)(0|1)(0|00)
    k1ao_aux1_4[A] += np.einsum("uvP, PQ, tRQ, RS, klS, vi, li -> tuk", int3c2e, int2c2e_inv, int2c2e_ip1[:, slcA], int2c2e_inv[slcA, :], int3c2e, mocc_2, mocc_2)
k1ao_aux1 = k1ao_aux1_1 + k1ao_aux1_2 + k1ao_aux1_3 + k1ao_aux1_4

## 基础工具：j2c 转换

PySCF 在处理梯度问题时，会引入函数 `_gen_metric_solve` (调用时以 `solve_j2c` 的形式出现)，以对辅助基指标作转换。

我们这里将要引入一个更复杂的函数。我们引入参数 `left` 与 `flip`。

一些提前的补充说明如下：

- 正常情况下，我们是要用 `overwrite_b = True` 以避免内存开销的。不过一方面这里是草稿，我们先不考虑这个问题；另一方面，在实际程序编写的时候，所有权问题是需要谨慎对待的。
- 这里的函数只保证对二维矩阵有效。比如 2c-2e ERI 导数矩阵的维度是 `tPQ`，这时用 `left = True` 会导致作用的维度是导数分量维度 `t`，而不是辅助基指标维度 `P`。
- 之所以要抽象出一个函数，是因为 j2c 分解不止有一种形式。它还有 eigenvalue 模式、以及 upper cholesky 分解 (REST 的 col-major 倾向) 模式。使用比较统一的函数接口，可以让我们在不同的分解模式下使用同样的代码。

In [51]:
def gen_solve_by_j2c(int2c):
    int2c_l = scipy.linalg.cholesky(int2c, lower=True)
    def solve_by_j2c(v, flip=False, left=True):
        res = None
        shape = v.shape
        if left and not flip:
            v = v.reshape(shape[0], -1)
            res = scipy.linalg.solve_triangular(int2c_l, v, lower=True).reshape(shape)
        elif left and flip:
            v = v.reshape(shape[0], -1)
            res = scipy.linalg.solve_triangular(int2c_l.T, v, lower=False).reshape(shape)
        elif not left and not flip:
            v = v.reshape(-1, shape[-1])
            res = scipy.linalg.solve_triangular(int2c_l.T, v.T, lower=False).T.reshape(shape)
        elif not left and flip:
            v = v.reshape(-1, shape[-1])
            res = scipy.linalg.solve_triangular(int2c_l, v.T, lower=True).T.reshape(shape)
        return res
    return solve_by_j2c

In [52]:
int2c2e_l_inv = scipy.linalg.inv(scipy.linalg.cholesky(int2c2e, lower=True))
solve_by_j2c = gen_solve_by_j2c(int2c2e)


- 默认情况下是 `left = True`, `flip = False`。典型的应用情景是对 3c-2e ERI $g_{P, \mu \nu}$ 转换到分解积分 $Y_{P, \mu \nu}$, $\mathbf{Y} = \mathbf{L}^{-1} \mathbf{g}$ 的情景。这里的 left 是指辅助基 $P$ 在被求解张量 $g_{P, \mu \nu}$ 的左边；flip 是指对于 Cholesky 类型分解 ($\mathbf{L}$ 并非是对称的)，我们不进行转置。

In [53]:
int3c2e_s2ij = _int3c_wrapper(mol, aux, "int3c2e", "s2ij")()
print(int3c2e_s2ij.shape)
assert np.allclose(solve_by_j2c(int3c2e_s2ij.T), mf.with_df._cderi)
assert np.allclose(int2c2e_l_inv @ int3c2e_s2ij.T, mf.with_df._cderi)

(1225, 131)


作为定义，在 Cholesky 下三角分解的情形下，有下述表达式成立：

In [54]:
assert np.allclose(solve_by_j2c(int3c2e_s2ij.T, left=True, flip=False), int2c2e_l_inv @ int3c2e_s2ij.T)
assert np.allclose(solve_by_j2c(int3c2e_s2ij.T, left=True, flip=True), int2c2e_l_inv.T @ int3c2e_s2ij.T)
assert np.allclose(solve_by_j2c(int3c2e_s2ij, left=False, flip=False), int3c2e_s2ij @ int2c2e_l_inv)
assert np.allclose(solve_by_j2c(int3c2e_s2ij, left=False, flip=True), int3c2e_s2ij @ int2c2e_l_inv.T)

- `left = False` 的情况会出现在 $(\partial_t P | Q)$ 的两侧都要作转换的情况。有时我们要作类似于下述的转换：$\partial \mathbf{J} \times \mathbf{J}^{-1} \times \partial \mathbf{J}$，为了数值稳定性我们倾向于使用 triangular solve。

In [55]:
np.allclose(
    solve_by_j2c(int2c2e_ip1[0], left=False, flip=True) @ solve_by_j2c(int2c2e_ip1[1], left=True, flip=False),
    np.einsum("PQ, QR, RS -> PS", int2c2e_ip1[0], int2c2e_inv, int2c2e_ip1[1])
)

True

- `flip = True` 的情形除了上面提到的类相似变换 (左右分别对应不翻转与翻转的两种情形)，另一个常用的情景是连续作两次转换 (即乘以 $\mathbf{J}^{-1}$)。出于数值稳定性，我们有意避免直接存储 $\mathbf{J}^{-1}$ 或分解的逆 ($\mathbf{L}^{-1}$)，而使用矩阵求解实现求逆。但普通矩阵求解通常是 LU 或类似较大开销算法，如果我们已经有 Cholesky 分解，那么就可以用 DTRSM 加快求解速度与降低内存开销。需要调用函数两次确实是麻烦了一些，但这大概是最优做法，也没有必要再抽象出一个新的函数出来。

## 电子积分的使用情况

我们首先列举所有出现过的电子积分，以及其对应的导数项。

| 编号 | 导数项 | 电子积分 (3c) | 电子积分 (2c) | 完成情况 |
|-----------|---------------------------------|-------------------|--------------|----|
| 20 - 1    | (10\|0) (0\|10)                 | `ip1`, `ip1`      |              | 待实战 |
| 20 - 2    | (11\|0) (0\|00)                 | `ipvip1`          |              | ✅ |
| 20 - 3    | (20\|0) (0\|00)                 | `ipip1`           |              | ✅ |
| 11 - 1    | (10\|1) (0\|0) (0\|00)          | `ip1ip2`          |              | ✅ |
| 11 - 2    | (10\|0) (0\|1) (0\|00)          | `ip1`             | `ip1`        | 待实战 |
| 11 - 3    | (10\|0) (1\|0) (0\|00)          | `ip1`             | `ip1`        | 待实战 |
| 11 - 4    | (10\|0) (0\|0) (1\|00)          | `ip1`, `ip2`      |              | 待实战 |
| 02 - 1    | (00\|2) (0\|00)                 | `ipip2`           |              | ✅ |
| 02 - 2    | (00\|0) (2\|0) (0\|00)          |                   | `ipip1`      | ✅ |
| 02 - 3a   | (00\|0) (1\|1) (0\|00)          |                   | `ip1ip2`     | ✅ |
| 02 - 3b   | (00\|0) (1\|0) (0\|1) (0\|00)   |                   | `ip1`        | ✅ |
| 02 - 4    | (00\|1) (1\|0) (0\|00)          | `ip2`             | `ip1`        | ✅ |
| 02 - 5    | (00\|1) (1\|00)                 | `ip2`, `ip2`      |              | ✅ |
| 02 - 6    | (00\|0) (0\|1) (1\|0) (0\|00)   |                   | `ip1`, `ip1` | ✅ |
| 02 - 7    | (00\|1) (0\|1) (0\|00)          | `ip2`             | `ip1`        | ✅ |
| 02 - 8    | (00\|0) (1\|0) (1\|0) (0\|00)   |                   | `ip1`, `ip1` | ✅ |
| f1ao_aux0 | (10\|0) (0\|00) and perm        | `ip1`             |              |  |
| f1ao_aux1 | (00\|1) (0\|00) and perm        | `ip2`             |              | ✅ |
| f1ao_aux1 | (00\|0) (1\|0) (0\|00) and perm |                   | `ip1`        | ✅ |

## 优化尝试：方针与补充约定

### 补充约定俗成

我们将在后续计算中，按照与 REST 相同的**内存排布顺序** (不是指维度，因此作为 row/col-major 不同的程序，通常维度刚好是相反的)。

在 PySCF 中，指标的顺序是

- 原子指标 `A, B`
- 分量指标 `t, s` (代表 x, y, z)
- 辅助基指标 `P`
- 占据轨道指标 `i`
- 原子轨道指标 `v, u`，**注意这里的顺序是反的，$\nu$ 在前 $\mu$ 在后**

在这里，Hessian 的指标顺序是 `A, B, t, s`，而在 REST 则应该是反过来的 `s, t, B, A`。

需要额外留意的是，对于类似于 ipvip1 的积分 $(\partial_t \mu \partial_s \nu | P)$，在这里的指标顺序是 `t, s, P, v, u`，而在 REST 则是 `u, v, P, s, t`。留意 ipvip1 的 `t` 对应 `u`，`s` 对应 `v`，但这个顺序在张量指标上是对称而不是对应的。

### 占据轨道重新定义

如前面的程序编写所述，我们将大量使用 `mocc_2`，即使用占据数加权后的占据轨道。

以后再出现轨道系数，我们就都使用这个概念了。

但唯一**需要注意**的地方是，在定义 Fock Skeleton 导数时，我们很可能要利用一次占据轨道优化，从而直接给出 f1mo 而非 f1ao。此时仍然需要使用没有占据数加权的轨道 `mocc`。

### 优化策略

- **内存优化**。首先要从内存优化角度入手。我们不希望像现在这样，所有的中间张量都被存储在内存中。

  内存控制的总原则是，完全避免 $n_\mathrm{basis}^2 n_\mathrm{aux}$ 级别内存消耗，接受 $n_\mathrm{occ}^2 n_\mathrm{aux}$ 级别的内存消耗。

  大多数项都满足这一条件；但其中有一个特例：`20 - 1` 的 K 积分贡献是唯一麻烦的一项。这一项很可能要求 $3 n_\mathrm{occ} n_\mathrm{basis} n_\mathrm{aux}$ 级别的内存消耗，否则会产生比较严重的额外计算开销。

- **电子积分调用优化**。我们要尽可能减少电子积分的调用次数。尽管 RI-JK 自洽场计算的电子积分耗时不多 (而矩阵乘法耗时更大)，但这单纯是因为我们假设体系不算太大，以至于电子积分全部可以存于内存，从而 $O(N^3)$ 电子积分只计算一次、$O(N^4)$ 矩阵乘法则需要计算多次。但梯度问题则是另一个情景：电子积分与矩阵乘法都只需要计算一次。即使计算复杂度更小，电子积分在梯度计算中的占比更大，耗时也更明显。因此，我们要采用的策略是，算一部分电子积分后，就尽可能蒿完它的羊毛，把相关的项都结算掉，不要留着下一次再算。

- **辅助基转换**。我们将使用类似于 PySCF 的 `solve_j2c` 函数 (由 `_gen_metric_solver` 得到)。但同时，我们也需要允许二次转换，以避免一次对导数张量的、经常是不必要的转换。该转换对于 Cholesky 分解的 J2C 而言需要是 inplace DTRSM 的。在 REST 中，对应的函数是 `get_solved_j3c`；但该函数有可能未来命名为 `solve_by_j2c`，且允许接受 `TensorMut` 类型。

### 新约定俗成下的电子积分

首先我们要作积分转置：

In [56]:
int3csort         = np.ascontiguousarray(np.einsum("  uvP ->   Pvu", int3c2e       ))
int3csort_ip1     = np.ascontiguousarray(np.einsum(" tuvP ->  tPvu", int3c2e_ip1   ))
int3csort_ip2     = np.ascontiguousarray(np.einsum(" tuvP ->  tPvu", int3c2e_ip2   ))
int3csort_ipvip1  = np.ascontiguousarray(np.einsum("tsuvP -> tsPvu", int3c2e_ipvip1))
int3csort_ipip1   = np.ascontiguousarray(np.einsum("tsuvP -> tsPvu", int3c2e_ipip1 ))
int3csort_ip1ip2  = np.ascontiguousarray(np.einsum("tsuvP -> tsPvu", int3c2e_ip1ip2))
int3csort_ipip2   = np.ascontiguousarray(np.einsum("tsuvP -> tsPvu", int3c2e_ipip2 ))
int2csort         = np.ascontiguousarray(np.einsum("   PQ ->    QP", int2c2e       ))
int2csort_ip1     = np.ascontiguousarray(np.einsum("  tPQ ->   tQP", int2c2e_ip1   ))
int2csort_ipip1   = np.ascontiguousarray(np.einsum(" tsPQ ->  tsQP", int2c2e_ipip1 ))
int2csort_ip1ip2  = np.ascontiguousarray(np.einsum(" tsPQ ->  tsQP", int2c2e_ip1ip2))

同时，我们要在可以的情况下，尽可能使用先前存好的、作过一次转换的电子积分 `cderi`，其维度是 `[naux, nao_tp]`。其中，`nao_tp` 是指 `nao * (nao + 1) / 2` 即三角对称的原子轨道指标维度。

In [57]:
cderi = mf.with_df._cderi
cderi.shape

(131, 1225)

我们纯粹为了程序方便，对 cderi 作三角矩阵展开；但实际程序实现中，这个过程必须对辅助基作分批处理。

In [58]:
cderi_ao = lib.unpack_tril(cderi)
cderi_ao.shape

(131, 49, 49)

### 常用中间变量

下述两个中间量，是我们可以接受在内存中全部存储的：

- `itm_j`: $\mathscr{J}_P = Y_{P, \nu \mu} D_{\nu \mu}$, shape `[naux]`
- `itm_k_occ`: $\mathscr{K}_{P, ji} = Y_{P, \nu \mu} C_{\nu j} C_{\mu i}$, shape `[naux, nocc, nocc]`

与一阶梯度情景不同，我们这里就不对 `itm_k_occ` 作三角矩阵压缩处理了。一方面后面的程序经常会用到，另一方面我们整体的程序需要接受一次 $3 n_\mathrm{occ} n_\mathrm{basis} n_\mathrm{aux}$ 的内存消耗，因而这里的 $n_\mathrm{occ}^2 n_\mathrm{aux}$ 内存消耗都不是重要的了。


In [59]:
itm_k_occ = np.einsum("Pvu, vj, ui -> Pji", cderi_ao, mocc_2, mocc_2)
itm_j = np.einsum("Puv, uv -> P", cderi_ao, dm0)

下面是暂时声明出来的临时变量；它们可以认为是内存控制好的。但由于算法优化还在进行，这些量是否最终采用将在后面的程序中决定。

In [60]:
solved_itm_j = solve_by_j2c(itm_j, left=True, flip=True)
solved_itm_k_occ = solve_by_j2c(itm_k_occ, left=True, flip=True)
solved_itm_k_aux = solved_itm_k_occ.reshape(naux, -1) @ solved_itm_k_occ.reshape(naux, -1).T

In [61]:
print(lib.fp(solved_itm_k_occ))

5.90999340397721


程序中会出现必须使用 $\mathbf{J}^{-1}$ 的情景。这种情况下，为了能兼顾接口与数值稳定性，我们对单位矩阵使用两次 `solve_by_j2c`，而不是直接使用 $\mathbf{J}^{-1}$。

In [62]:
int2c_inv = solve_by_j2c(solve_by_j2c(np.eye(naux), left=True, flip=True), left=False, flip=False)
assert np.allclose(int2c_inv, int2c2e_inv)

## 优化尝试 (1)：非 3c-2e 导数贡献

非 3c-2e 导数，其优化相对容易，因为我们大多数时候不需要考虑 3c-2e 电子积分是否要计算、如何分批。不少计算过程都是一步到位的。我们也先从非 3c-2e 导数贡献开始优化。

### Skeleton 02-2 J

In [63]:
# trial 01
dbas_J02_2_recap = np.einsum("P, PQ, tsRQ, SR, S -> tsQ", itm_j, int2c2e_l_inv, int2csort_ipip1, int2c2e_l_inv, itm_j)
assert np.allclose(dbas_J02_2_recap, dbas_J02_2, rtol=1e-4, atol=1e-6)

# trial 02
dbas_J02_2_recap = np.einsum("P, tsPQ, Q -> tsP", solved_itm_j, int2csort_ipip1, solved_itm_j)
assert np.allclose(dbas_J02_2_recap, dbas_J02_2, rtol=1e-4, atol=1e-6)

### Skeleton 02-2 K

In [64]:
# trial 01
dbas_K02_2_recap = np.einsum("Pji, PQ, tsQR, SR, Sji -> tsQ", itm_k_occ, int2c2e_l_inv, int2c2e_ipip1, int2c2e_l_inv, itm_k_occ)
assert np.allclose(dbas_K02_2_recap, dbas_K02_2, rtol=1e-4, atol=1e-6)

# trial 02
dbas_K02_2_recap = np.einsum("Qji, tsQR, Rji -> tsQ", solved_itm_k_occ, int2c2e_ipip1, solved_itm_k_occ)
assert np.allclose(dbas_K02_2_recap, dbas_K02_2, rtol=1e-4, atol=1e-6)

# trial 03
dbas_K02_2_recap = np.einsum("PQ, tsPQ -> tsQ", solved_itm_k_aux, int2csort_ipip1)
assert np.allclose(dbas_K02_2_recap, dbas_K02_2, rtol=1e-4, atol=1e-6)

### Skeleton 02-3a J

In [65]:
dbas_J02_3a_recap = np.einsum("P, tsPQ, Q -> tsPQ", solved_itm_j, int2csort_ip1ip2, solved_itm_j)
assert np.allclose(dbas_J02_3a_recap, dbas_J02_3a, rtol=1e-4, atol=1e-6)

### Skeleton 02-3a K

In [66]:
dbas_K02_3a_recap = np.einsum("PQ, tsPQ -> tsPQ", solved_itm_k_aux, int2csort_ip1ip2)
assert np.allclose(dbas_K02_3a_recap, dbas_K02_3a, rtol=1e-4, atol=1e-6)

### Skeleton 02-3b J

In [67]:
# trial 01
dbas_J02_3b_recap = np.einsum("Q, tQR, RS, sST, T -> tsQT", solved_itm_j, int2csort_ip1, int2c2e_inv, int2csort_ip1, solved_itm_j)
assert np.allclose(dbas_J02_3b_recap, dbas_J02_3b, rtol=1e-4, atol=1e-6)

# trial 02
# This utilizes int2c2e_ip1 is asymmetric tensor
# so a negative sign is needed for fliping the axes
assert np.allclose(int2csort_ip1, - int2csort_ip1.swapaxes(-2, -1))
rsolved_int2csort_ip1 = np.asarray([solve_by_j2c(m, left=False, flip=True) for m in int2csort_ip1])
rrsolved_int2csort_ip1 = np.asarray([solve_by_j2c(m, left=False, flip=False) for m in rsolved_int2csort_ip1])
lsolved_int2csort_ip1 = np.asarray([solve_by_j2c(m, left=True, flip=False) for m in int2csort_ip1])
llsolved_int2csort_ip1 = np.asarray([solve_by_j2c(m, left=True, flip=True) for m in lsolved_int2csort_ip1])
dbas_J02_3b_recap = - np.einsum("P, tPR, sQR, Q -> tsPQ", solved_itm_j, rsolved_int2csort_ip1, rsolved_int2csort_ip1, solved_itm_j)
assert np.allclose(dbas_J02_3b_recap, dbas_J02_3b, rtol=1e-4, atol=1e-6)

### Skeleton 02-3b K

In [68]:
dbas_K02_3b_recap = - np.einsum("PQ, tPR, sQR -> tsPQ", solved_itm_k_aux, rsolved_int2csort_ip1, rsolved_int2csort_ip1)
assert np.allclose(dbas_K02_3b_recap, dbas_K02_3b, rtol=1e-4, atol=1e-6)

### Skeleton 02-6 J

后面的一些 aux 2nd 导数没有办法，必须要使用 $\mathbf{J}^{-1}$。

In [69]:
dbas_J02_6_recap = np.einsum("R, tRP, PQ, sSQ, S -> tsPQ", solved_itm_j, int2csort_ip1, int2c_inv, int2csort_ip1, solved_itm_j)
assert np.allclose(dbas_J02_6_recap, dbas_J02_6, rtol=1e-4, atol=1e-6)

### Skeleton 02-6 K

In [70]:
dbas_K02_6_recap = np.einsum("RS, tRP, PQ, sSQ -> tsPQ", solved_itm_k_aux, int2csort_ip1, int2c_inv, int2csort_ip1)
assert np.allclose(dbas_K02_6_recap, dbas_K02_6, rtol=1e-4, atol=1e-6)

### Skeleton 02-8 J

尽管 02-8 的例子是可以对 ip1 导数矩阵作两次 solve 给出的，但这样还是麻烦了一点。我们就按照与 02-6 相同的处理方式，仍然直接乘以 int2c_inv。

In [71]:
dbas_J02_8_recap = np.einsum("R, tPR, PS, sQS, Q -> tsPQ", solved_itm_j, int2csort_ip1, int2c_inv, int2csort_ip1, solved_itm_j)
assert np.allclose(dbas_J02_8_recap, dbas_J02_8, rtol=1e-4, atol=1e-6)

### Skeleton 02-8 K

In [72]:
dbas_K02_8_recap = np.einsum("PS, tPR, RQ, sQS -> tsPQ", solved_itm_k_aux, int2c2e_ip1, int2c2e_inv, int2c2e_ip1)
assert np.allclose(dbas_K02_8_recap, dbas_K02_8, rtol=1e-4, atol=1e-6)

### J1ao aux1-3/4: no n^3 additional memory

In [73]:
j1ao_aux1_3_recap = np.zeros([natm, 3, nao, nao])
j1ao_aux1_4_recap = np.zeros([natm, 3, nao, nao])

# all `Puv, AtP -> Atuv` can use triangular-packed cderi.

# temporary area for j1 aux1-3
tmp1 = np.einsum("tRQ, R -> tQ", int2csort_ip1, solved_itm_j)
tmp2 = np.zeros((natm, 3, naux))
for A in range(mol.natm):
    _, _, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    tmp2[A, :, slcA] = tmp1[:, slcA]
tmp3 = solve_by_j2c(tmp2, left=False, flip=True)
j1ao_aux1_3_recap = np.einsum("Puv, AtP -> Atuv", cderi_ao, tmp3)

# temporary area for j1 aux1-4
tmp1 = lsolved_int2csort_ip1 * solved_itm_j
tmp2 = np.zeros((natm, 3, naux))
for A in range(mol.natm):
    _, _, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    tmp2[A] = tmp1[:, :, slcA].sum(axis=-1)
j1ao_aux1_4_recap = np.einsum("Puv, AtP -> Atuv", cderi_ao, tmp2)

assert np.allclose(j1ao_aux1_3_recap, j1ao_aux1_3, rtol=1e-4, atol=1e-6)
assert np.allclose(j1ao_aux1_4_recap, j1ao_aux1_4, rtol=1e-4, atol=1e-6)

### K1bra aux1-3/4

In [74]:
occ_invsqrt = occ_occupation ** -0.5

In [75]:
# trial 01

k1bra_aux1_3_recap = np.zeros([natm, 3, nocc, nao])
k1bra_aux1_4_recap = np.zeros([natm, 3, nocc, nao])

for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    k1bra_aux1_3_recap[A] += np.einsum("Qij, tRQ, PR, Pkl, lj, i -> tik", solved_itm_k_occ[slcA], int2csort_ip1[:, :, slcA], int2c2e_l_inv, cderi_ao, mocc_2, occ_invsqrt)
    k1bra_aux1_4_recap[A] += np.einsum("Qij, tQR, PR, Pkl, lj, i -> tik", solved_itm_k_occ, int2csort_ip1[:, :, slcA], int2c2e_l_inv[:, slcA], cderi_ao, mocc_2, occ_invsqrt)

assert np.allclose(k1bra_aux1_3_recap, mocc.T @ k1ao_aux1_3, rtol=1e-4, atol=1e-6)
assert np.allclose(k1bra_aux1_4_recap, mocc.T @ k1ao_aux1_4, rtol=1e-4, atol=1e-6)

In [76]:
# trial 02

k1bra_aux1_3_recap = np.zeros([natm, 3, nocc, nao])
k1bra_aux1_4_recap = np.zeros([natm, 3, nocc, nao])

cderi_xob = np.einsum("Pvu, vi -> Piu", cderi_ao, mocc_2)
solved_cderi_xob = solve_by_j2c(cderi_xob, left=True, flip=True)

for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    k1bra_aux1_3_recap[A] += np.einsum("Qij, tRQ, Rjk, i -> tik", solved_itm_k_occ[slcA], int2csort_ip1[:, :, slcA], solved_cderi_xob, occ_invsqrt)
    k1bra_aux1_4_recap[A] += np.einsum("Qij, tQR, Rjk, i -> tik", solved_itm_k_occ, int2csort_ip1[:, :, slcA], solved_cderi_xob[slcA], occ_invsqrt)

assert np.allclose(k1bra_aux1_3_recap, mocc.T @ k1ao_aux1_3, rtol=1e-4, atol=1e-6)
assert np.allclose(k1bra_aux1_4_recap, mocc.T @ k1ao_aux1_4, rtol=1e-4, atol=1e-6)

In [77]:
# trial 03

k1bra_aux1_3_recap = np.zeros([natm, 3, nocc, nao])
k1bra_aux1_4_recap = np.zeros([natm, 3, nocc, nao])

# additional memory: cderi_xob (aux * basis * occ)
# this memory should be reused after solve_by_j2c
cderi_xob = np.einsum("Pvu, vi -> Piu", cderi_ao, mocc_2)
# can discard cderi_xob here
solved_cderi_xob = solve_by_j2c(cderi_xob, left=True, flip=True)

# handle aux1_3 term
# additional memory: tmp1 (aux * basis * occ)
k1bra_aux1_3_recap = np.zeros([natm, 3, nocc, nao])
for t in range(3):
    tmp1 = np.einsum("RQ, Rjk -> Qjk", int2csort_ip1[t], solved_cderi_xob)
    for A in range(mol.natm):
        _, _, p0, p1 = auxslices[A]
        slcA = slice(p0, p1)
        k1bra_aux1_3_recap[A, t] = np.einsum("Qji, Qjk -> ik", solved_itm_k_occ[slcA], tmp1[slcA])
k1bra_aux1_3_recap *= occ_invsqrt[None, None, :, None]

# handle aux1_4 term
# additional memory: tmp1 (aux * occ * occ)
k1bra_aux1_4_recap = np.zeros([natm, 3, nocc, nao])
for t in range(3):
    tmp1 = np.einsum("Qij, QR -> Rij", solved_itm_k_occ, int2csort_ip1[t])
    for A in range(mol.natm):
        _, _, p0, p1 = auxslices[A]
        slcA = slice(p0, p1)
        k1bra_aux1_4_recap[A, t] = np.einsum("Rji, Rjk -> ik", tmp1[slcA], solved_cderi_xob[slcA])
k1bra_aux1_4_recap *= occ_invsqrt[None, None, :, None]

# most memory consuming part is k1bra_aux1_3, in total 2 * aux * basis * occ

assert np.allclose(k1bra_aux1_3_recap, mocc.T @ k1ao_aux1_3, rtol=1e-4, atol=1e-6)
assert np.allclose(k1bra_aux1_4_recap, mocc.T @ k1ao_aux1_4, rtol=1e-4, atol=1e-6)

## 优化尝试 (2)：单纯 3c-2e-ip2

其次是单纯 3c-2e-ip2 的导数贡献。3c-2e-ip2 的导数是对辅助基进行的，而辅助基的计算本身是可以分批的。如果没有出现非常复杂的辅助基转换，那么尽管这里必须考虑分批，但分批相对容易。

我们在原型程序实现时，可能就不明确引入分批。

### Skeleton 02-4 J

In [78]:
# int3csort_ip2_aux can be evaluated with batch
int3csort_ip2_aux = np.einsum("tPuv, uv -> tP", int3csort_ip2, dm0)
dbas_J02_4_recap = np.zeros((3, 3, naux, naux))
tmp1 = np.einsum("sRQ, R -> sQ", int2csort_ip1, solved_itm_j)
for t in range(3):
    dbas_J02_4_recap[t] = np.einsum("P, PQ, sQ -> sPQ", int3csort_ip2_aux[t], int2c_inv, tmp1)
assert np.allclose(dbas_J02_4_recap, dbas_J02_4, rtol=1e-4, atol=1e-6)

### Skeleton 02-4 K

**NOTE**: OUTPUT-OCC-OCC (ip2)

In [79]:
# trial 01
dbas_K02_4_recap = np.einsum("tPuv, PQ, sRQ, Rij, ui, vj -> tsPQ", int3csort_ip2, int2c2e_inv, int2csort_ip1, solved_itm_k_occ, mocc_2, mocc_2)
assert np.allclose(dbas_K02_4_recap, dbas_K02_4, rtol=1e-4, atol=1e-6)

# trial 02
# memory cost: 3 * aux * occ * occ
# compute cost: 3 * aux * basis^2 * occ + 3 * aux * basis * occ^2
int3csort_ip2_occ = np.zeros((3, naux, nocc, nocc))
for t in range(3):
    int3csort_ip2_occ[t] = np.einsum("Puv, ui, vj -> Pij", int3csort_ip2[t], mocc_2, mocc_2)

for s in range(3):
    # memory cost: aux * occ * occ
    # compute cost: 3 * aux^2 * occ^2
    tmp1 = np.einsum("RQ, Rij -> Qij", int2csort_ip1[s], solved_itm_k_occ)
    # batch P, evaluate int3csort_ip2 in batch (not shown here)
    for t in range(3):
        tmp2 = np.einsum("Pij, Qij -> PQ", int3csort_ip2_occ[t], tmp1)
        dbas_K02_4_recap[t, s] = tmp2 * int2c_inv

assert np.allclose(dbas_K02_4_recap, dbas_K02_4, rtol=1e-4, atol=1e-6)

### Skeleton 02-5 J

我们在 02-4 中已经得到了 `int3csort_ip2_aux` (`tP`)。这里我们利用这个中间量。

In [80]:
dbas_J02_5_recap = np.einsum("tP, PQ, sQ -> tsPQ", int3csort_ip2_aux, int2c_inv, int3csort_ip2_aux)
assert np.allclose(dbas_J02_5_recap, dbas_J02_5, rtol=1e-4, atol=1e-6)

### Skeleton 02-5 K

在 02-4 中我们已经存好所有的 `int3csort_ip2_occ` (`tPij`)，那么这里就可以利用 02-4 的中间量。

In [81]:
tmp1 = np.einsum("tPij, sQij -> tsPQ", int3csort_ip2_occ, int3csort_ip2_occ)
dbas_K02_5_recap = tmp1 * int2c_inv
assert np.allclose(dbas_K02_5_recap, dbas_K02_5, rtol=1e-4, atol=1e-6)

### Skeleton 02-7 J

In [82]:
dbas_J02_7_recap = np.einsum("tP, sPR, R -> tsPR", int3csort_ip2_aux, llsolved_int2csort_ip1, solved_itm_j)
assert np.allclose(dbas_J02_7_recap, dbas_J02_7, rtol=1e-4, atol=1e-6)

### Skeleton 02-7 K

In [83]:
tmp1 = np.einsum("tPij, Rij -> tPR", int3csort_ip2_occ, solved_itm_k_occ)
dbas_K02_7_recap = np.einsum("tPR, sPR -> tsPR", tmp1, llsolved_int2csort_ip1)
assert np.allclose(dbas_K02_7_recap, dbas_K02_7, rtol=1e-4, atol=1e-6)

### J1ao aux1-1/2

In [84]:
j1ao_aux1_1_recap = np.zeros([natm, 3, nao, nao])
j1ao_aux1_2_recap = np.zeros([natm, 3, nao, nao])

# j1ao_aux1_1
for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    j1ao_aux1_1_recap[A] -= np.einsum("tPuv, P -> tuv", int3csort_ip2[:, slcA], solved_itm_j[slcA])

# j1ao_aux1_2
tmp1 = np.zeros((natm, 3, naux))
for A in range(mol.natm):
    _, _, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    tmp1[A, :, slcA] = int3csort_ip2_aux[:, slcA]
tmp2 = solve_by_j2c(tmp1, left=False, flip=True)
j1ao_aux1_2_recap = - np.einsum("Puv, AtP -> Atuv", cderi_ao, tmp2)

assert np.allclose(j1ao_aux1_1_recap, j1ao_aux1_1, rtol=1e-4, atol=1e-6)
assert np.allclose(j1ao_aux1_2_recap, j1ao_aux1_2, rtol=1e-4, atol=1e-6)

### K1bra aux1-1/2

In [85]:
# naive
k1bra_aux1_1_recap = np.zeros([natm, 3, nocc, nao])
k1bra_aux1_2_recap = np.zeros([natm, 3, nocc, nao])

for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    k1bra_aux1_1_recap[A] -= np.einsum("tuvP, PQ, klQ, vj, lj, ui, i -> tik", int3c2e_ip2[:, :, :, slcA], int2c2e_inv[slcA, :], int3c2e, mocc_2, mocc_2, mocc_2, occ_invsqrt)
    k1bra_aux1_2_recap[A] -= np.einsum("uvP, PQ, tklQ, vj, lj, ui, i -> tik", int3c2e, int2c2e_inv[:, slcA], int3c2e_ip2[:, :, :, slcA], mocc_2, mocc_2, mocc_2, occ_invsqrt)

assert np.allclose(k1bra_aux1_1_recap, mocc.T @ k1ao_aux1_1, rtol=1e-4, atol=1e-6)
assert np.allclose(k1bra_aux1_2_recap, mocc.T @ k1ao_aux1_2, rtol=1e-4, atol=1e-6)

In [86]:
# trial 01
k1bra_aux1_1_recap = np.zeros([natm, 3, nocc, nao])
k1bra_aux1_2_recap = np.zeros([natm, 3, nocc, nao])

for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    k1bra_aux1_1_recap[A] -= np.einsum("tPij, Pjk, i -> tik", int3csort_ip2_occ[:, slcA], solved_cderi_xob[slcA], occ_invsqrt)
    k1bra_aux1_2_recap[A] -= np.einsum("Pij, tPkl, lj, i -> tik", solved_itm_k_occ[slcA], int3csort_ip2[:, slcA], mocc_2, occ_invsqrt)

assert np.allclose(k1bra_aux1_1_recap, mocc.T @ k1ao_aux1_1, rtol=1e-4, atol=1e-6)
assert np.allclose(k1bra_aux1_2_recap, mocc.T @ k1ao_aux1_2, rtol=1e-4, atol=1e-6)

### 简单小结

3c-2e-ip2 的情景要分为两种：ipip2 与 ip1 + ip2 (Skeleton 2nd) 或 ip2 (Fock 1nd)。

其余的部分首先要分批计算得到 `int3csort_ip2_aux` (`tP`) 与 `int3csort_ip2_occ` (`tPij`) 并完全存储于内存。这些分量得到后，后面的计算就容易很多 (通常是比较简单的类 einsum 计算，实际实现时再转为矩阵乘法，不需要再分批了)。分批计算除了上面两者外，还有 `j1ao_aux1_1` 要同时处理。

## 优化尝试 (3)：一次性电子积分

一次性电子积分的计算模式会比较单一，且同时处理可以节省一些计算量。

我们总结其中一种 K 积分计算模式。该模式只有在多个贡献同时计算时，才能真正有效地减少计算量。

该模式的 einsum 策略是

```python
APuv, Pij, ui, vj -> AP / Au / Auv
```

该模式的伪代码计算是

```python
for P in batch:
    Pij, ui, vj -> Puv
    APuv, Puv -> AP / Au / Auv
```

该策略与通常的 einsum 不同，它需要先从占据轨道空间的 `Pij` 张量，张成更大的 `Puv` 张量。这一步看起来是内存损耗的，但我们是可以对辅助基指标 `P` 分批处理的，因此这里的内存用量可控。

否则，对于类似于 ipip2 的任务，我们需要先对 `APuv, ui, vj -> APij` 作压缩；但这种压缩其实是耗费更多计算量的 (相比于前面的做法多了 `A - 1` 倍)。

未来一定会遇到多种电子积分同时计算的情况。如果一类 `Puv` 可以共用，那么张开空间的做法只需要一次即可，不需要每个电子积分都再搞一次。伪代码是

```python
for P in batch:
    extract_Pij_batch(itm_k_occ / solved_itm_k_occ)
    Pij, ui, vj -> Puv
    compute_intor_in_batch(APuv)
    APuv, Puv -> AP / Au / Auv
    compute_intor_in_batch(BPuv)
    BPuv, Puv -> BP / Bu / Buv
```

In [87]:
solved_itm_k_ao = np.einsum("Pij, ui, vj -> Puv", solved_itm_k_occ, mocc_2, mocc_2)

### Skeleton 02-1

In [88]:
tmp1 = np.einsum("tsPuv, uv -> tsP", int3csort_ipip2, dm0)
dbas_J02_1_recap = np.einsum("tsP, P -> tsP", tmp1, solved_itm_j)
assert np.allclose(dbas_J02_1_recap, dbas_J02_1, rtol=1e-4, atol=1e-6)

In [89]:
dbas_K02_1_recap = np.einsum("tsPuv, Puv -> tsP", int3csort_ipip2, solved_itm_k_ao)
assert np.allclose(dbas_K02_1_recap, dbas_K02_1, rtol=1e-4, atol=1e-6)

### Skeleton 20-2

In [90]:
# notice uv <-> vu
# this is not that important, and can be changed when counting contribution of `de`
tmp1 = np.einsum("tsPvu, P -> tsvu", int3csort_ipvip1, solved_itm_j)
dbas_J20_2_recap = np.einsum("tsvu, uv -> tsuv", tmp1, dm0)
assert np.allclose(dbas_J20_2_recap, dbas_J20_2, rtol=1e-4, atol=1e-6)

In [91]:
dbas_K20_2_recap = np.einsum("tsPvu, Pvu -> tsuv", int3csort_ipvip1, solved_itm_k_ao)
assert np.allclose(dbas_K20_2_recap, dbas_K20_2, rtol=1e-4, atol=1e-6)

### Skeleton 20-3

In [92]:
tmp1 = np.einsum("tsPvu, P -> tsvu", int3csort_ipip1, solved_itm_j)
dbas_J20_3_recap = np.einsum("tsvu, uv -> tsuv", tmp1, dm0)
assert np.allclose(dbas_J20_3_recap, dbas_J20_3, rtol=1e-4, atol=1e-6)

In [93]:
dbas_K20_3_recap = np.einsum("tsPvu, Pvu -> tsuv", int3csort_ipip1, solved_itm_k_ao)
assert np.allclose(dbas_K20_3_recap, dbas_K20_3, rtol=1e-4, atol=1e-6)

### Skeleton 11-1

In [94]:
dbas_J11_1_recap = np.einsum("tsPvu, uv, P -> tsuP", int3csort_ip1ip2, dm0, solved_itm_j)
assert np.allclose(dbas_J11_1_recap, dbas_J11_1, rtol=1e-4, atol=1e-6)

In [95]:
dbas_K11_1_recap = np.einsum("tsPvu, Puv -> tsuP", int3csort_ip1ip2, solved_itm_k_ao)
assert np.allclose(dbas_K11_1_recap, dbas_K11_1, rtol=1e-4, atol=1e-6)

## 优化尝试 (4 K)：3c-2e-ip1 K

ip1 积分的最麻烦之处，在于 K 积分计算时，至少有一个原子轨道指标需要保留。我们在 20-1 与 11-2/3 将采用不同的策略实现该计算。

In [96]:
int3csort_ip1_xob = np.einsum("tPvu, vi -> tPiu", int3csort_ip1, mocc_2)
solved_int3csort_ip1_xob = np.array([solve_by_j2c(m, left=True, flip=False) for m in int3csort_ip1_xob])
solved_itm_k_xob = np.einsum("Pji, uj -> Piu", solved_itm_k_occ, mocc_2)

### Skeleton 11-4 K

In [117]:
# this term requires another time of solve to solved_int3csort_ip1_xob.
# dbas_K11_4_recap = np.einsum("tPju, PQ, sQji, ui -> tsuQ", solved_int3csort_ip1_xob, int2c2e_l_inv, int3csort_ip2_occ, mocc_2)
dbas_K11_4_recap = np.zeros((3, 3, nao, naux))
for t in range(3):
    tmp1 = solve_by_j2c(solved_int3csort_ip1_xob[t], left=True, flip=True)
    dbas_K11_4_recap[t] = np.einsum("Pju, sPji, ui -> suP", tmp1, int3csort_ip2_occ, mocc_2)
np.allclose(dbas_K11_4_recap, dbas_K11_4, rtol=1e-4, atol=1e-6)

True

### Skeleton 20-1 K

In [97]:
dbas_K20_1a_recap = np.einsum("tPju, sPjv, ui, ki -> tsuv", solved_int3csort_ip1_xob, solved_int3csort_ip1_xob, mocc_2, mocc_2)
de_K20_1a_recap = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_1a_recap[A, B] += 2 * np.einsum("tsuv -> ts", dbas_K20_1a_recap[:, :, p0A:p1A, p0B:p1B])
np.allclose(de_K20_1a_recap, de_K20_1a, rtol=1e-4, atol=1e-6)

False

### Skeleton 11-2 K

In [98]:
solved_int3csort_ip1_atm_aux = np.zeros((natm, 3, naux, naux))
for A in range(mol.natm):
    _, _, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    solved_int3csort_ip1_atm_aux[A] = np.einsum("tPiu, Qiu -> tQP", solved_int3csort_ip1_xob[..., slcA], solved_itm_k_xob[..., slcA])

In [99]:
dbas_K11_2_recap = np.einsum("tPju, sRP, Rju -> tsuR", solved_int3csort_ip1_xob, rsolved_int2csort_ip1, solved_itm_k_xob)
assert np.allclose(dbas_K11_2_recap, dbas_K11_2, rtol=1e-4, atol=1e-6)

In [100]:
tmp1 = np.einsum("AtRP, sRP -> AtsR", solved_int3csort_ip1_atm_aux, rsolved_int2csort_ip1)
de_K11_2_recap = np.zeros([natm, natm, 3, 3])
for B in range(natm):
    _, _, p0, p1 = auxslices[B]
    slcB = slice(p0, p1)
    de_K11_2_recap[:, B] = 2 * np.einsum("AtsR -> Ats", tmp1[..., slcB])
de_K11_2_recap += de_K11_2_recap.transpose(1, 0, 3, 2)
np.allclose(de_K11_2_recap, de_K11_2, rtol=1e-4, atol=1e-6)

True

### Skeleton 11-3 K

In [101]:
dbas_K11_3_recap = np.einsum("tPju, PQ, sRQ, Rju -> tsuQ", int3csort_ip1_xob, int2c2e_inv, int2csort_ip1, solved_itm_k_xob)
assert np.allclose(dbas_K11_3_recap, dbas_K11_3, rtol=1e-4, atol=1e-6)

In [102]:
# `AtRP, PQ` step can be performed by solve_by_j2c inplace, and then int3csort_ip1_atm_aux will be destroyed.
tmp1 = np.einsum("AtRP, PQ, sRQ -> AtsQ", solved_int3csort_ip1_atm_aux, int2c2e_l_inv, int2csort_ip1)
de_K11_3_recap = np.zeros([natm, natm, 3, 3])
for B in range(natm):
    _, _, p0, p1 = auxslices[B]
    slcB = slice(p0, p1)
    de_K11_3_recap[:, B] = -2 * np.einsum("AtsQ -> Ats", tmp1[..., slcB])
de_K11_3_recap += de_K11_3_recap.transpose(1, 0, 3, 2)
assert np.allclose(de_K11_3_recap, de_K11_3, rtol=1e-4, atol=1e-6)

In [170]:
# another way, which should used together with 11-4 term.

dbas_K11_3_recap = np.zeros((3, 3, nao, naux))
for t in range(3):
    tmp1 = solve_by_j2c(solved_int3csort_ip1_xob[t], left=True, flip=True)
    dbas_K11_3_recap[t] = np.einsum("Qju, sRQ, Rju -> suQ", tmp1, int2csort_ip1, solved_itm_k_xob)
assert np.allclose(dbas_K11_3_recap, dbas_K11_3, rtol=1e-4, atol=1e-6)

### K1bra aux0

In [104]:
# baseline
scr1 = np.einsum("tPiu, Pik -> tuk", solved_int3csort_ip1_xob, cderi_xob)

k1ao_aux0_1_recap = np.zeros([natm, 3, nao, nao])
k1ao_aux0_2_recap = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    k1ao_aux0_1_recap[A, :, slcA, :] -= scr1[:, slcA, :]
    k1ao_aux0_1_recap[A, :, :, slcA] -= scr1[:, slcA, :].swapaxes(-1, -2)
    
k1bra_aux0_1_recap = np.zeros([natm, 3, nocc, nao])
k1bra_aux0_1_recap = mocc.T @ k1ao_aux0_1_recap

assert np.allclose(k1bra_aux0_1_recap, mocc.T @ k1ao_aux0_1, rtol=1e-4, atol=1e-6)

In [105]:
# trial 01

k1ao_aux0_2_recap = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    scr2 = np.einsum("tPkl, QP, Qiu, li -> tku", int3csort_ip1[..., slcA], int2c2e_l_inv, cderi_xob, mocc_2[slcA])
    k1ao_aux0_2_recap[A] -= scr2 + scr2.swapaxes(-1, -2)

k1bra_aux0_2_recap = np.zeros([natm, 3, nocc, nao])
k1bra_aux0_2_recap = mocc.T @ k1ao_aux0_2_recap
assert np.allclose(k1bra_aux0_2_recap, mocc.T @ k1ao_aux0_2, rtol=1e-4, atol=1e-6)

In [106]:
# trial 02

k1bra_aux0_2_recap = np.zeros([natm, 3, nocc, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    scr2 = np.einsum("tPkl, QP, Qju, lj, ki -> tiu", int3csort_ip1[..., slcA], int2c2e_l_inv, cderi_xob, mocc_2[slcA], mocc)
    k1bra_aux0_2_recap[A] -= scr2
    scr2 = np.einsum("tPkl, QP, Qju, lj, ui -> tik", int3csort_ip1[..., slcA], int2c2e_l_inv, cderi_xob, mocc_2[slcA], mocc)
    k1bra_aux0_2_recap[A] -= scr2

assert np.allclose(k1bra_aux0_2_recap, mocc.T @ k1ao_aux0_2, rtol=1e-4, atol=1e-6)

In [107]:
# trial 03

k1bra_aux0_2_recap = np.zeros([natm, 3, nocc, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    scr2 = np.einsum("tPil, Pju, lj, i -> tiu", solved_int3csort_ip1_xob[..., slcA], cderi_xob, mocc_2[slcA], occ_invsqrt)
    k1bra_aux0_2_recap[A] -= scr2
    scr2 = np.einsum("tPkl, Pji, lj, i -> tik", int3csort_ip1[..., slcA], solved_itm_k_occ, mocc_2[slcA], occ_invsqrt)
    k1bra_aux0_2_recap[A] -= scr2

assert np.allclose(k1bra_aux0_2_recap, mocc.T @ k1ao_aux0_2, rtol=1e-4, atol=1e-6)

## 优化尝试 (4 J)：3c-2e-ip1 J

In [146]:
int3csort_ip1_aux = np.einsum("tPvu, vu -> tPu", int3csort_ip1, dm0)
lsolved_int3csort_ip1_aux = np.array([solve_by_j2c(m, left=True, flip=False) for m in int3csort_ip1_aux])
llsolved_int3csort_ip1_aux = np.array([solve_by_j2c(m, left=True, flip=True) for m in lsolved_int3csort_ip1_aux])

### Skeleton 11-4 J

In [147]:
dbas_J11_4_recap = np.einsum("tQu, sQ -> tsuQ", llsolved_int3csort_ip1_aux, int3csort_ip2_aux)
np.allclose(dbas_J11_4_recap, dbas_J11_4, rtol=1e-4, atol=1e-6)

True

### Skeleton 20-1 J

In [150]:
dbas_J20_1_recap = np.einsum("tPu, sPk -> tsuk", lsolved_int3csort_ip1_aux, lsolved_int3csort_ip1_aux)
np.allclose(dbas_J20_1_recap, dbas_J20_1, rtol=1e-4, atol=1e-6)

True

### Skeleton 11-2 J

In [162]:
dbas_J11_2_recap = - np.einsum("tPu, sPR, R -> tsuR", lsolved_int3csort_ip1_aux, lsolved_int2csort_ip1, solved_itm_j)
np.allclose(dbas_J11_2_recap, dbas_J11_2, rtol=1e-4, atol=1e-6)

True

### Skeleton 11-3 J

In [167]:
dbas_J11_3_recap = np.einsum("tQu, sRQ, R -> tsuQ", llsolved_int3csort_ip1_aux, int2csort_ip1, solved_itm_j)
np.allclose(dbas_J11_3_recap, dbas_J11_3, rtol=1e-4, atol=1e-6)

True